In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### **Data Reading**

In [0]:
df=spark.read.format('delta')\
    .load('abfss://bronze@adventwork.dfs.core.windows.net/productCategories')

### **Deleting _rescued_data column**

In [0]:
df=df.drop("_rescued_data")

### **Schema**

In [0]:
df.printSchema()

root
 |-- ProductCategoryKey: string (nullable = true)
 |-- CategoryName: string (nullable = true)



### **Changing ProductCategoryKey columns' format**

In [0]:
df=df.withColumn('ProductCategoryKey',col("ProductCategoryKey").cast('int'))

### **ProductCategoryKey column's Data Quality**

In [0]:
duplicate_keys = (

    df.groupBy("ProductCategoryKey")
      .count()
      .filter(col("count") > 1)
      .select("ProductCategoryKey")

)

In [0]:
invalid_df = (

    df.filter(col("ProductCategoryKey").isNull())
      .unionByName(
          df.join(
              duplicate_keys,on="ProductCategoryKey",how="inner"
          )

      )

)

In [0]:
invalid_df.write.mode('append')\
    .save('abfss://silver@adventwork.dfs.core.windows.net/quarantine/productCategories')

### **Data Writing**

In [0]:
if spark.catalog.tableExists('adventure_works.silver.productCategories'):
    df_silver_category = spark.read.table('adventure_works.silver.productCategories')
    df=df.join(df_silver_category,'ProductCategoryKey','left_anti')


In [0]:
df.write.format('delta').mode('append')\
    .save('abfss://silver@adventwork.dfs.core.windows.net/productCategories')

In [0]:
%sql
create table if not exists adventure_works.silver.productcategories
using delta
location 'abfss://silver@adventwork.dfs.core.windows.net/productCategories'

In [0]:
df.display()

ProductCategoryKey,CategoryName


In [0]:
%sql
select * from adventure_works.silver.productcategories

ProductCategoryKey,CategoryName
1,Bikes
2,Components
3,Clothing
4,Accessories
